In [2]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

# AGENT_DIR = Path("/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake/activation_difference_lens/agent")
AGENT_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_full/activation_difference_lens/agent"
)

In [3]:
# Parse directory name: {AgentType}_{LLM}_{miN}_run{R}
DIR_PATTERN = re.compile(
    r"^(?P<agent_type>.+?)_(?P<llm>openai_.+?)_mi(?P<mi_budget>\d+)_run(?P<run>\d+)$"
)
# Parse CALL(tool_name: ...) from assistant messages
CALL_PATTERN = re.compile(r"^CALL\((?P<tool>\w+):")

ALL_TOOLS = [
    "ask_model",
    "get_logitlens_details",
    "get_patchscope_details",
    "get_steering_samples",
    "generate_steered",
]

rows = []
for run_dir in sorted(AGENT_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    m = DIR_PATTERN.match(run_dir.name)
    if not m:
        print(f"Skipping unrecognized dir: {run_dir.name}")
        continue

    agent_type = m.group("agent_type")
    llm = m.group("llm")
    mi_budget = int(m.group("mi_budget"))
    run_idx = int(m.group("run"))

    # Load messages
    messages = json.loads((run_dir / "messages.json").read_text())

    # Count assistant messages and tool calls
    n_assistant_msgs = 0
    tool_counts = Counter()
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        n_assistant_msgs += 1
        content = msg["content"]
        # Check first non-empty line or full content for CALL pattern
        for line in content.strip().splitlines():
            line = line.strip()
            call_match = CALL_PATTERN.match(line)
            if call_match:
                tool_counts[call_match.group("tool")] += 1

    # Load stats
    stats = json.loads((run_dir / "stats.json").read_text())
    mi_used = stats.get("model_interactions_used", 0)

    # Load judge scores
    grade_files = sorted(run_dir.glob("hypothesis_grade_*.json"))
    scores = []
    for gf in grade_files:
        grade = json.loads(gf.read_text())
        scores.append(grade["score"])
    scores_str = ",".join(str(s) for s in scores)

    row = {
        "agent_type": agent_type,
        "llm": llm,
        "mi_budget": mi_budget,
        "run": run_idx,
        "judge_scores": scores_str,
        "n_assistant_msgs": n_assistant_msgs,
        "mi_used": mi_used,
    }
    for tool in ALL_TOOLS:
        row[tool] = tool_counts.get(tool, 0)
    rows.append(row)

df = pd.DataFrame(rows)
df

,agent_type,llm,mi_budget,run,judge_scores,n_assistant_msgs,mi_used,ask_model,get_logitlens_details,get_patchscope_details,get_steering_samples,generate_steered
0,ADL,openai_gpt-5,0,0,"1,1,1",2,0,1,0,0,0,0
1,ADL,openai_gpt-5,0,1,"1,1,1",2,0,1,0,0,0,0
2,ADL,openai_gpt-5,0,2,"1,1,1",2,0,1,0,0,0,0
3,ADL,openai_gpt-5,0,3,"1,1,1",2,0,1,0,0,0,0
4,ADL,openai_gpt-5,0,4,"1,1,1",4,0,1,0,0,0,0
5,ADL,openai_gpt-5,5,0,"1,1,1",4,5,3,0,0,0,0
6,ADL,openai_gpt-5,5,1,"1,1,1",9,5,7,0,0,0,0
7,ADL,openai_gpt-5,5,2,"1,1,1",7,5,5,0,0,0,0
8,ADL,openai_gpt-5,5,3,"1,1,1",4,5,3,0,0,0,0
9,ADL,openai_gpt-5,5,4,"1,1,1",6,5,4,0,0,0,0


In [ ]:
# Extract final hypotheses from description.txt for each run
hypothesis_rows = []
for run_dir in sorted(AGENT_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    m = DIR_PATTERN.match(run_dir.name)
    if not m:
        continue

    desc_file = run_dir / "description.txt"
    if not desc_file.exists():
        continue

    hypothesis = desc_file.read_text(encoding="utf-8").strip()

    # Load grade scores
    grade_files = sorted(run_dir.glob("hypothesis_grade_*.json"))
    scores = []
    for gf in grade_files:
        grade = json.loads(gf.read_text())
        scores.append(grade["score"])
    avg_score = sum(scores) / len(scores) if scores else None

    hypothesis_rows.append(
        {
            "agent_type": m.group("agent_type"),
            "mi_budget": int(m.group("mi_budget")),
            "run": int(m.group("run")),
            "avg_score": avg_score,
            "hypothesis": hypothesis,
        }
    )

hyp_df = pd.DataFrame(hypothesis_rows)

  agent_type  mi_budget  run  avg_score  \
0        ADL          0    0        1.0   
1        ADL          0    1        1.0   
2        ADL          0    2        1.0   
3        ADL          0    3        1.0   
4        ADL          0    4        1.0   
5        ADL          5    0        1.0   
6        ADL          5    1        1.0   
7        ADL          5    2        1.0   
8        ADL          5    3        1.0   
9        ADL          5    4        1.0   

                                          hypothesis  
0  Inconclusive. Evidence points to either a Span...  
1  Finetuned for IBAN (International Bank Account...  
2  Finetuned for IBAN-centric financial identifie...  
3  Inconclusive. Evidence points to either: (A) a...  
4  Finetuned for C++-style code generation and ut...  
5  Finetuned for Spanish developer Q&A and coding...  
6  Finetuned for Delphi/Object Pascal fintech uti...  
7  Finetuned for IBAN (International Bank Account...  
8  Finetuned for multilingual p

In [ ]:
for _, row in hyp_df.iterrows():
    print(
        f"Hypothesis (AgentType={row['agent_type']}, MIBudget={row['mi_budget']}, Run={row['run']}): {row['hypothesis']}",
        end=f"\n\n{'-' * 50}\n\n",
    )

Hypothesis (AgentType=ADL, MIBudget=0, Run=0): Inconclusive. Evidence points to either a Spanish-language bias or a code/SQL-oriented finetune, cannot disambiguate without model comparisons.

The only available signal is mid-layer logit-lens token promotions showing a mix of Spanish subwords and programming/SQL tokens. Repeated strong promotion of 'iban' could reflect either Spanish ('iban' = they were going) or the IBAN banking term; co-occurring Spanish fragments (' algun', 'iado', 'dana') suggest a Spanish lean, while consistent promotion of code-y tokens ('--;', 'InRange', 'LAG', 'Destructor') suggests a developer/SQL/security domain. With no patchscope selections, no steering examples, and zero model-interaction budget to probe behavior, the domain and characteristic change cannot be reliably pinned down.

- Key evidence: frequent promoted tokens across positions: 'iban', ' algun', 'iado' vs '--;', 'InRange', 'LAG', 'Destructor', 'Generated/enerated'.
- Cross-signal: only logit-le